# AI Tutor Local Notebook: AKT Training + Evaluation Graphs

This local Windows/PC notebook runs the same AKT-first baseline against the local project files. It validates the local `My_AI_Tutor_Datasets` folder before training, writes a local notebook config, supports checkpoint recovery, and saves evaluation graphs under `AI Tutor/outputs/figures`.


In [ ]:
# ============================================================
# 0. Local user settings
# ============================================================

from pathlib import Path

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'AI Tutor').exists() and ((candidate / 'Datasets (Organized)').exists() or (candidate / 'My_AI_Tutor_Datasets').exists()):
            return candidate
        if candidate.name == 'AI Tutor' and (candidate / 'src').exists():
            return candidate.parent
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
BASE_DIR = PROJECT_ROOT / 'AI Tutor'
DATASET_ROOT = PROJECT_ROOT / 'My_AI_Tutor_Datasets'
DOWNLOAD_DIR = PROJECT_ROOT / 'downloaded_datasets'
ORGANIZED_DATASETS_CANDIDATES = [
    PROJECT_ROOT / 'Datasets (Organized)',
    PROJECT_ROOT / 'Dataset (Organized)',
    DOWNLOAD_DIR / 'Datasets (Organized)',
    DOWNLOAD_DIR / 'Dataset (Organized)',
]
ORGANIZED_DATASETS_DIR = next((p for p in ORGANIZED_DATASETS_CANDIDATES if p.exists()), ORGANIZED_DATASETS_CANDIDATES[0])

RUN_AKT_EDNET_TRAINING = True
FORCE_AKT_RETRAIN = False  # Set True only when you intentionally want to rerun the full EdNet training.
RUN_RAG_BUILD = True
RUN_TUTOR_DEMO = False
RUN_ONNX_EXPORT = True
RUN_XAI = True
RUN_FINAL_EVAL = True
REFRESH_AKT_GRAPH_DATA = False  # False reuses cached graph predictions; missing or stale cache is built automatically.
RUN_BENCHMARK = False
RUN_EVAL_GRAPHS = True

VERIFY_DATASETS_BEFORE_CONTINUE = True
REQUIRE_FULL_MY_AI_TUTOR_DATASET = False  # Set True only when you want the inventory check to fail on missing optional datasets.

KT_SOURCE_MODE = 'ednet'
REQUIRE_ALL_SELECTED_KT_SOURCES = True

DATASET_PATH_OVERRIDE = ''

Continue_Last_checkpoint = True
AUTO_RESUME = Continue_Last_checkpoint
RESUME_CHECKPOINT = ''

MAX_ROWS = None
EPOCHS = 20
BATCH_SIZE = None

TUTOR_MODE = 'turboquant'
OPENAI_MODEL = 'gpt-4.1-mini'
LOCAL_LLM_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
LOCAL_LLM_QUANTIZATION = 'turboquant'
TURBOQUANT_BASE_URL = None
TURBOQUANT_BITS = 4
TURBOVEC_BITS = 3

print('Project root:', PROJECT_ROOT)
print('Code root:', BASE_DIR)
print('Raw dataset root:', DATASET_ROOT)
print('Organized datasets dir:', ORGANIZED_DATASETS_DIR)


In [ ]:
# ============================================================
# 1. Dependency check
# ============================================================

import importlib.util
import subprocess
import sys

INSTALL_MISSING = False  # Set True if you want this cell to install missing packages.

REQUIRED = {
    'yaml': 'pyyaml',
    'pandas': 'pandas',
    'numpy': 'numpy',
    'torch': 'torch',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
}
OPTIONAL = {
    'pyarrow': 'pyarrow',
    'onnx': 'onnx',
    'onnxruntime': 'onnxruntime',
    'sentence_transformers': 'sentence-transformers',
    'chromadb': 'chromadb',
    'openai': 'openai',
    'transformers': 'transformers',
    'pypdf': 'pypdf',
    'docx': 'python-docx',
    'shap': 'shap',
}

def has_module(name):
    return importlib.util.find_spec(name) is not None

def pip_install(packages):
    if not packages:
        return
    print('Installing:', packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--upgrade-strategy', 'only-if-needed', *packages])

missing_required = [pkg for mod, pkg in REQUIRED.items() if not has_module(mod)]
missing_optional = [pkg for mod, pkg in OPTIONAL.items() if not has_module(mod)]

if missing_required:
    print('Missing required packages:', missing_required)
    if INSTALL_MISSING:
        pip_install(missing_required)
if missing_optional:
    print('Optional packages missing:', missing_optional)
    print('Optional packages are needed only for RAG/LLM/ONNX/SHAP cells.')

if missing_required and not INSTALL_MISSING:
    raise ImportError('Missing required packages. Set INSTALL_MISSING=True and rerun this cell, or install them in your environment.')

print('Dependency check complete.')


In [ ]:
# ============================================================
# 2. Dataset inventory and EdNet KT dataset discovery
# ============================================================

import json
import os
from pathlib import Path
import yaml

ORGANIZED_DATASETS_CANDIDATES = [Path(p) for p in globals().get('ORGANIZED_DATASETS_CANDIDATES', [
    PROJECT_ROOT / 'Datasets (Organized)',
    PROJECT_ROOT / 'Dataset (Organized)',
    PROJECT_ROOT / 'downloaded_datasets' / 'Datasets (Organized)',
    PROJECT_ROOT / 'downloaded_datasets' / 'Dataset (Organized)',
])]
ORGANIZED_DATASETS_DIR = next((p for p in ORGANIZED_DATASETS_CANDIDATES if p.exists()), ORGANIZED_DATASETS_CANDIDATES[0])


def dedupe_paths(paths):
    out, seen = [], set()
    for p in paths:
        p = Path(p)
        try:
            key = str(p.resolve()) if p.exists() else str(p)
        except Exception:
            key = str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out


def existing_first(candidates, default=None):
    for candidate in candidates:
        p = Path(candidate)
        if p.exists():
            return p.resolve()
    if default is not None:
        return Path(default)
    raise FileNotFoundError('None of these dataset candidates exist:\n' + '\n'.join(str(c) for c in candidates))


def organized_candidates(*parts):
    return [root.joinpath(*parts) for root in ORGANIZED_DATASETS_CANDIDATES]


def organized_file_candidates(folder, *relative_parts):
    candidates = []
    for root in ORGANIZED_DATASETS_CANDIDATES:
        base = root / folder
        candidates.append(base.joinpath(*relative_parts))
        if relative_parts and base.exists():
            try:
                candidates.extend(p for p in base.rglob(relative_parts[-1]) if p.is_file())
            except OSError as exc:
                print(f'Skipping organized scan under {base}: {exc}')
    return dedupe_paths(candidates)


EDNET_CONTENTS_CANDIDATES = (
    organized_file_candidates('EdNet_Contents', 'contents', 'questions.csv') +
    organized_file_candidates('EdNet_Contents', 'questions.csv') +
    organized_file_candidates('My_AI_Tutor_Datasets_RAW', 'EdNet-Contents', 'contents', 'questions.csv') +
    [
        PROJECT_ROOT / 'My_AI_Tutor_Datasets' / 'EdNet-Contents' / 'contents' / 'questions.csv',
        PROJECT_ROOT / 'Data stuff' / 'EdNet-Contents' / 'contents' / 'questions.csv',
    ]
)
EDNET_KT1_CANDIDATES = organized_candidates('EdNet_KT1', 'KT1') + organized_candidates('EdNet_KT1') + [PROJECT_ROOT / 'My_AI_Tutor_Datasets' / 'KT1', PROJECT_ROOT / 'Data stuff' / 'EdNet-KT1']
EDNET_KT2_CANDIDATES = organized_candidates('EdNet_KT2') + [PROJECT_ROOT / 'My_AI_Tutor_Datasets' / 'KT2', PROJECT_ROOT / 'Data stuff' / 'EdNet-KT2']
EDNET_KT3_CANDIDATES = organized_candidates('EdNet_KT3') + [PROJECT_ROOT / 'My_AI_Tutor_Datasets' / 'KT3', PROJECT_ROOT / 'Data stuff' / 'EdNet-KT3']
EDNET_KT4_CANDIDATES = organized_candidates('EdNet_KT4') + [PROJECT_ROOT / 'My_AI_Tutor_Datasets' / 'KT4', PROJECT_ROOT / 'Data stuff' / 'EdNet-KT4']
EDNET_KT34_CANDIDATES = organized_candidates('EdNet_KT34') + [PROJECT_ROOT / 'downloaded_datasets' / 'kagglehub' / 'ednet_kt34_feather_csv', PROJECT_ROOT / 'downloaded_datasets' / 'kagglehub' / 'ednet_kt34']
RAG_DOCS_CANDIDATES = organized_candidates('RAG_Datasets')

PROJECT_DATASET_REQUIREMENTS = [
    {'name': 'EdNet contents questions.csv', 'kind': 'file', 'required': True, 'candidates': EDNET_CONTENTS_CANDIDATES},
    {'name': 'EdNet KT1 folder', 'kind': 'dir', 'required': True, 'candidates': EDNET_KT1_CANDIDATES},
    {'name': 'EdNet KT2 folder', 'kind': 'dir', 'required': True, 'candidates': EDNET_KT2_CANDIDATES},
    {'name': 'EdNet KT3 folder', 'kind': 'dir', 'required': True, 'candidates': EDNET_KT3_CANDIDATES},
    {'name': 'EdNet KT4 folder', 'kind': 'dir', 'required': True, 'candidates': EDNET_KT4_CANDIDATES},
    {'name': 'Old EdNet KT34 converted/supervised folder', 'kind': 'dir', 'required': False, 'candidates': EDNET_KT34_CANDIDATES},
    {'name': 'RAG datasets/sources folder', 'kind': 'dir', 'required': True, 'candidates': RAG_DOCS_CANDIDATES},
]


def validate_project_datasets():
    if not VERIFY_DATASETS_BEFORE_CONTINUE:
        print('Dataset validation skipped by setting.')
        return {}
    inventory = {}
    missing = []
    print('Local project EdNet dataset inventory check:')
    print('Organized datasets dir:', ORGANIZED_DATASETS_DIR)
    for req in PROJECT_DATASET_REQUIREMENTS:
        found = None
        for candidate in dedupe_paths(req['candidates']):
            if req.get('kind') == 'dir' and candidate.is_dir():
                found = candidate.resolve()
                break
            if req.get('kind') == 'file' and candidate.is_file() and candidate.stat().st_size > 0:
                found = candidate.resolve()
                break
        inventory[req['name']] = str(found) if found else None
        if found:
            extra = f' ({found.stat().st_size / (1024 ** 2):.1f} MB)' if found.is_file() else ''
            print(f" OK   {req['name']}: {found}{extra}")
        else:
            print(f" MISS {req['name']}")
            if req.get('required', False):
                missing.append(req['name'])
    if missing:
        raise FileNotFoundError('Missing required local datasets: ' + ', '.join(missing))
    return inventory


def discover_dataset_path():
    if DATASET_PATH_OVERRIDE:
        p = Path(DATASET_PATH_OVERRIDE)
        if p.exists():
            return p.resolve()
        raise FileNotFoundError(f'DATASET_PATH_OVERRIDE does not exist: {p}')
    return existing_first(EDNET_CONTENTS_CANDIDATES)


DATASET_INVENTORY = validate_project_datasets()
DATASET_PATH = discover_dataset_path()
RAG_DOCS_PATH = existing_first(RAG_DOCS_CANDIDATES)
print('Using EdNet metadata:', DATASET_PATH)
print('Using RAG docs/sources:', RAG_DOCS_PATH)


In [ ]:
# ============================================================
# 3. Write local EdNet-only notebook config
# ============================================================

import os
import yaml
import torch
from pathlib import Path

CUDA_AVAILABLE = torch.cuda.is_available()
batch_size = BATCH_SIZE if BATCH_SIZE is not None else (96 if CUDA_AVAILABLE else 32)
CONFIG_PATH = BASE_DIR / 'configs' / 'local_notebook.yaml'

ORGANIZED_DATASETS_CANDIDATES = [Path(p) for p in globals().get('ORGANIZED_DATASETS_CANDIDATES', [
    PROJECT_ROOT / 'Datasets (Organized)',
    PROJECT_ROOT / 'Dataset (Organized)',
    PROJECT_ROOT / 'downloaded_datasets' / 'Datasets (Organized)',
    PROJECT_ROOT / 'downloaded_datasets' / 'Dataset (Organized)',
])]
ORGANIZED_DATASETS_DIR = next((p for p in ORGANIZED_DATASETS_CANDIDATES if p.exists()), ORGANIZED_DATASETS_CANDIDATES[0])


def existing_first(candidates, default=None):
    for candidate in candidates:
        p = Path(candidate)
        if p.exists():
            return p.resolve()
    if default is not None:
        return Path(default)
    raise FileNotFoundError('None of these dataset candidates exist:\n' + '\n'.join(str(c) for c in candidates))


def organized_candidates(*parts):
    return [str(root.joinpath(*parts)) for root in ORGANIZED_DATASETS_CANDIDATES]


def organized_file_candidates(folder, *relative_parts):
    candidates = []
    for root in ORGANIZED_DATASETS_CANDIDATES:
        base = root / folder
        candidates.append(str(base.joinpath(*relative_parts)))
        if relative_parts and base.exists():
            try:
                candidates.extend(str(p) for p in base.rglob(relative_parts[-1]) if p.is_file())
            except OSError as exc:
                print(f'Skipping organized scan under {base}: {exc}')
    return list(dict.fromkeys(candidates))


EDNET_CONTENTS_CANDIDATES = (
    organized_file_candidates('EdNet_Contents', 'contents', 'questions.csv') +
    organized_file_candidates('EdNet_Contents', 'questions.csv') +
    organized_file_candidates('My_AI_Tutor_Datasets_RAW', 'EdNet-Contents', 'contents', 'questions.csv') +
    [
        str(PROJECT_ROOT / 'My_AI_Tutor_Datasets/EdNet-Contents/contents/questions.csv'),
        str(PROJECT_ROOT / 'Data stuff/EdNet-Contents/contents/questions.csv'),
    ]
)
DATASET_PATH = existing_first(EDNET_CONTENTS_CANDIDATES, default=DATASET_PATH)

KT_SOURCES = [
    {
        'name': 'ednet_kt3',
        'type': 'ednet_folder',
        'path_candidates': organized_candidates('EdNet_KT3') + [
            str(PROJECT_ROOT / 'My_AI_Tutor_Datasets/KT3'),
            str(PROJECT_ROOT / 'Data stuff/EdNet-KT3'),
        ],
        'correct_answer_path_candidates': EDNET_CONTENTS_CANDIDATES,
        'enabled': True,
    },
    {
        'name': 'ednet_kt4',
        'type': 'ednet_folder',
        'path_candidates': organized_candidates('EdNet_KT4') + [
            str(PROJECT_ROOT / 'My_AI_Tutor_Datasets/KT4'),
            str(PROJECT_ROOT / 'Data stuff/EdNet-KT4'),
        ],
        'correct_answer_path_candidates': EDNET_CONTENTS_CANDIDATES,
        'enabled': True,
        'required': False,
    },
    {
        'name': 'ednet_kt2_uploaded',
        'type': 'ednet_folder',
        'path_candidates': organized_candidates('EdNet_KT2') + [
            str(PROJECT_ROOT / 'My_AI_Tutor_Datasets/KT2'),
            str(PROJECT_ROOT / 'Data stuff/EdNet-KT2'),
        ],
        'correct_answer_path_candidates': EDNET_CONTENTS_CANDIDATES,
        'enabled': True,
    },
    {
        'name': 'ednet_kt1',
        'type': 'ednet_folder',
        'path_candidates': organized_candidates('EdNet_KT1', 'KT1') + organized_candidates('EdNet_KT1') + [
            str(PROJECT_ROOT / 'My_AI_Tutor_Datasets/KT1'),
            str(PROJECT_ROOT / 'Data stuff/EdNet-KT1'),
        ],
        'correct_answer_path_candidates': EDNET_CONTENTS_CANDIDATES,
        'enabled': True,
        'required': True,
    },
]

KT_SOURCE_MODE = str(globals().get('KT_SOURCE_MODE', 'ednet')).strip().lower()


def select_kt_sources(mode):
    aliases = {
        'all': 'ednet',
        'full': 'ednet',
        'multi': 'ednet',
        'multi_source': 'ednet',
        'ednet_only': 'ednet',
    }
    mode = aliases.get(mode, mode)
    require_selected = bool(globals().get('REQUIRE_ALL_SELECTED_KT_SOURCES', True))
    if mode == 'ednet':
        return mode, 'ednet_kt1_kt2_kt3_kt4', True, KT_SOURCES, require_selected
    raise ValueError("KT_SOURCE_MODE must be 'ednet'.")


KT_SOURCE_MODE, KT_DATASET_NAME, USE_MULTI_SOURCE_KT, SELECTED_KT_SOURCES, REQUIRE_ALL_KT_SOURCES = select_kt_sources(KT_SOURCE_MODE)

RAG_DOCS_CANDIDATES = organized_candidates('RAG_Datasets')
RAG_DOCS_PATH = existing_first(RAG_DOCS_CANDIDATES)

config = {
    'environment': 'local-notebook-ednet-only',
    'seed': 42,
    'paths': {
        'data_root': str(ORGANIZED_DATASETS_DIR),
        'dataset_path': str(DATASET_PATH),
        'dataset_path_candidates': EDNET_CONTENTS_CANDIDATES,
        'processed_dir': str(BASE_DIR / 'outputs' / 'processed'),
        'checkpoints_dir': str(BASE_DIR / 'checkpoints'),
        'outputs_dir': str(BASE_DIR / 'outputs'),
        'logs_dir': str(BASE_DIR / 'logs'),
        'rag_docs_dir': str(RAG_DOCS_PATH),
        'rag_index_dir': str(BASE_DIR / 'outputs' / 'rag_index_datasets'),
        'retrieval_log': str(BASE_DIR / 'outputs' / 'retrieval_logs.jsonl'),
        'tutor_log': str(BASE_DIR / 'outputs' / 'tutor_logs.jsonl'),
        'explanations_dir': str(BASE_DIR / 'outputs' / 'explanations'),
    },
    'data': {
        'dataset_name': KT_DATASET_NAME,
        'use_multi_source': USE_MULTI_SOURCE_KT,
        'kt_sources': SELECTED_KT_SOURCES,
        'max_rows': MAX_ROWS,
        'column_overrides': {},
        'min_seq_len': 3,
        'max_seq_len': 200,
        'stride': 200,
        'require_all_kt_sources': REQUIRE_ALL_KT_SOURCES,
        'ednet_progress_every_files': 1000,
        'processed_cache_path': 'outputs/processed/kt_processed_ednet_only.pt',
        'cache_sources': True,
        'cache_processed': True,
        'rebuild_cache': False,
        'split': {'train': 0.8, 'val': 0.1, 'test': 0.1},
    },
    'model': {'type': 'akt', 'd_model': 128, 'n_heads': 8, 'n_layers': 3, 'dropout': 0.2, 'ff_multiplier': 4},
    'training': {
        'epochs': EPOCHS,
        'batch_size': batch_size,
        'lr': 0.001,
        'weight_decay': 0.01,
        'patience': 5,
        'num_workers': 0 if os.name == 'nt' else (2 if CUDA_AVAILABLE else 0),
        'mixed_precision': CUDA_AVAILABLE,
        'grad_clip': 1.0,
        'scheduler': 'cosine',
        'save_every_epoch': True,
        'pin_memory': CUDA_AVAILABLE,
    },
    'rag': {
        'backend': 'turbovec',
        'collection_name': 'ai_tutor_rag_datasets_local_v2',
        'embedding_backend': 'auto',
        'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
        'turbovec_bits': TURBOVEC_BITS,
        'hashing_features': 1024,
        'chunk_size': 900,
        'chunk_overlap': 150,
        'top_k': 4,
        'rebuild_index': False,
        'max_file_mb': 50,
        'max_files': 120,
        'max_chunks': 25000,
        'max_chars_per_file': 500000,
    },
    'tutor': {
        'mode': TUTOR_MODE,
        'openai_model': OPENAI_MODEL,
        'local_model': LOCAL_LLM_MODEL,
        'turboquant_model': LOCAL_LLM_MODEL,
        'turboquant_base_url': TURBOQUANT_BASE_URL,
        'turboquant_bits': TURBOQUANT_BITS,
        'max_new_tokens': 220,
        'temperature': 0.4,
        'quantization': LOCAL_LLM_QUANTIZATION,
        'allow_direct_answer': False,
        'hint_mode': True,
    },
    'evaluation': {'rag_top_k': 4, 'notes': 'Local EdNet-only AKT + RAG + Socratic tutor baseline using Datasets (Organized) when available.'},
    'benchmark': {'akt_batches': 20, 'onnx_batches': 50, 'run_local_llm': False, 'run_api_llm': False},
}
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('CUDA available:', CUDA_AVAILABLE)
if CUDA_AVAILABLE:
    print('GPU:', torch.cuda.get_device_name(0))
print('Organized datasets dir:', ORGANIZED_DATASETS_DIR)
print('RAG docs dir:', RAG_DOCS_PATH)
print('KT source mode:', KT_SOURCE_MODE)
print('Enabled KT sources:', ', '.join(src['name'] for src in SELECTED_KT_SOURCES))
print(CONFIG_PATH)
print(CONFIG_PATH.read_text(encoding='utf-8')[:5000])


In [ ]:
# ============================================================
# 4. AKT EdNet
# ============================================================

import os
import subprocess
import sys
from pathlib import Path


def run_streaming(cmd, cwd):
    """Run a command and print child-process output live in the notebook."""
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    env['PYTHONFAULTHANDLER'] = '1'
    env['PYTHONUTF8'] = '1'
    print('Running:', ' '.join(str(x) for x in cmd), flush=True)
    process = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding='utf-8',
        errors='replace',
        bufsize=1,
        env=env,
    )
    tail = []
    try:
        assert process.stdout is not None
        for line in process.stdout:
            tail.append(line.rstrip())
            tail[:] = tail[-40:]
            print(line, end='', flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        print('\nStopping child training process...', flush=True)
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    if code != 0:
        exit_hint = f'exit code {code}'
        if os.name == 'nt' and code > 0:
            exit_hint += f' / Windows 0x{code:08X}'
        print('\nTraining process failed with ' + exit_hint + '.', flush=True)
        if tail:
            print('Last child-process output:', flush=True)
            print('\n'.join(tail[-20:]), flush=True)
        raise RuntimeError(
            'AKT training failed before completion. The traceback line in run_streaming is only the wrapper; '
            'read the last child-process output above for the real cause.'
        )
    return code


def _safe_checkpoint_candidates() -> list[Path]:
    """Find top-level checkpoints for the current EdNet run only."""
    roots: list[Path] = []

    def add_root(path):
        if path is None:
            return
        p = Path(path)
        if p not in roots:
            roots.append(p)

    add_root(BASE_DIR / 'checkpoints')
    add_root(Path.cwd() / 'checkpoints')
    add_root(BASE_DIR.parent / 'checkpoints')

    found: list[Path] = []
    for root in roots:
        if not root.exists():
            continue
        try:
            for name in ('last_checkpoint.pt', 'best_model.pt'):
                direct = root / name
                if direct.exists():
                    found.append(direct)
        except OSError as exc:
            print(f'Skipping checkpoint scan under {root}: {exc}')

    unique: dict[str, Path] = {}
    for path in found:
        try:
            if path.exists() and path.is_file():
                unique[str(path.resolve())] = path
        except OSError:
            continue
    return list(unique.values())


def _load_training_config():
    cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8')) or {}
    data_cfg = cfg.get('data', {}) or {}
    sources = [src for src in data_cfg.get('kt_sources', []) if isinstance(src, dict) and src.get('enabled', True)]
    return cfg, data_cfg, sources


def _assert_training_source_config():
    _cfg, data_cfg, sources = _load_training_config()
    requested_mode = str(globals().get('KT_SOURCE_MODE', '')).strip().lower()
    source_names = [str(src.get('name', '<unnamed>')) for src in sources]
    required_source_names = [str(src.get('name', '<unnamed>')) for src in sources if src.get('required', True)]
    print('Training config data.use_multi_source:', bool(data_cfg.get('use_multi_source', False)))
    print('Training config data.require_all_kt_sources:', bool(data_cfg.get('require_all_kt_sources', False)))
    print('Training config enabled KT sources:', ', '.join(source_names) if source_names else 'primary dataset only')
    print('Training config required KT sources:', ', '.join(required_source_names) if required_source_names else 'none')
    if requested_mode in {'ednet', 'all'}:
        if not data_cfg.get('use_multi_source') or not sources:
            raise RuntimeError("KT_SOURCE_MODE is EdNet-only, but local_notebook.yaml is not multi-source. Rerun the local user settings cell and the config cell before training.")
        if not data_cfg.get('require_all_kt_sources'):
            raise RuntimeError("KT_SOURCE_MODE is EdNet-only, but data.require_all_kt_sources is false. Rerun the config cell before training.")
    return data_cfg, required_source_names


def _print_training_source_summary(expected_source_names):
    summary_path = BASE_DIR / 'outputs' / 'akt_training_summary.json'
    if not summary_path.exists():
        print('Training summary not found yet:', summary_path)
        return
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('Training summary dataset_path:', summary.get('dataset_path'))
    source_counts = summary.get('source_counts') or {}
    if source_counts:
        print('Training summary source_counts:')
        for name, count in source_counts.items():
            print(f'  {name}: {int(count):,}')
    else:
        print('Training summary source_counts: primary dataset only / not multi-source')
    requested_mode = str(globals().get('KT_SOURCE_MODE', '')).strip().lower()
    if requested_mode in {'ednet', 'all'}:
        missing = [name for name in expected_source_names if int(source_counts.get(name, 0) or 0) <= 0]
        if missing:
            raise RuntimeError('Training did not include all selected KT sources with nonzero sequences: ' + ', '.join(missing))


def _training_already_complete(data_cfg, expected_source_names) -> bool:
    """Skip expensive EdNet retraining when the existing artefacts already match this config."""
    if bool(globals().get('FORCE_AKT_RETRAIN', False)):
        return False
    outputs_dir = BASE_DIR / 'outputs'
    checkpoints_dir = BASE_DIR / 'checkpoints'
    summary_path = outputs_dir / 'akt_training_summary.json'
    best_path = checkpoints_dir / 'best_model.pt'
    last_path = checkpoints_dir / 'last_checkpoint.pt'
    if not summary_path.exists() or not (best_path.exists() or last_path.exists()):
        return False
    try:
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
    except Exception as exc:
        print('Existing training summary could not be read; training will run again:', exc)
        return False
    source_counts = summary.get('source_counts') or {}
    missing = [name for name in expected_source_names if int(source_counts.get(name, 0) or 0) <= 0]
    epochs_required = int((_load_training_config()[0].get('training') or {}).get('epochs', 20))
    epochs_done = int(summary.get('epochs_ran') or 0)
    if missing:
        print('Existing training summary is missing required sources, so training will run again:', ', '.join(missing))
        return False
    if epochs_done < epochs_required:
        print(f'Existing training summary has {epochs_done}/{epochs_required} epochs, so training will run again.')
        return False
    print(f'AKT training already completed ({epochs_done}/{epochs_required} epochs) and matching checkpoints exist.')
    print('Skipping the heavy EdNet training cell. Set FORCE_AKT_RETRAIN = True before this cell to retrain deliberately.')
    _print_training_source_summary(expected_source_names)
    return True


resume_args = []
continue_last_checkpoint = bool(globals().get('Continue_Last_checkpoint', globals().get('AUTO_RESUME', True)))
if continue_last_checkpoint:
    if RESUME_CHECKPOINT:
        resume_path = Path(RESUME_CHECKPOINT)
        if resume_path.exists():
            resume_args = ['--resume', str(resume_path)]
        else:
            print('Requested RESUME_CHECKPOINT was not found:', resume_path)
    else:
        candidates = _safe_checkpoint_candidates()
        if candidates:
            candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
            print('Continue_Last_checkpoint=True; resuming from:', candidates[0])
            resume_args = ['--resume', str(candidates[0])]
        else:
            print('Continue_Last_checkpoint=True but no checkpoint was found; starting fresh.')
else:
    print('Continue_Last_checkpoint=False; starting from scratch and writing new checkpoints.')

if RUN_AKT_EDNET_TRAINING:
    data_cfg, expected_source_names = _assert_training_source_config()
    if not _training_already_complete(data_cfg, expected_source_names):
        cmd = [sys.executable, '-u', 'train_akt.py', '--config', str(CONFIG_PATH), *resume_args]
        run_streaming(cmd, cwd=BASE_DIR)
        _print_training_source_summary(expected_source_names)
else:
    print('Skipping AKT EdNet training.')


In [ ]:
# ============================================================
# 6. Required RAG index and tutor demo
# ============================================================

import subprocess
import sys
from pathlib import Path

import yaml

cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8')) or {}
rag_docs_dir = Path(cfg.get('paths', {}).get('rag_docs_dir', ''))
if not rag_docs_dir.is_dir():
    raise FileNotFoundError(f'Required RAG_Datasets folder not found: {rag_docs_dir}')

rag_files = [p for p in rag_docs_dir.rglob('*') if p.is_file()]
if not rag_files:
    raise FileNotFoundError(f'Required RAG_Datasets folder is empty: {rag_docs_dir}')

print(f'RAG docs dir OK: {rag_docs_dir} ({len(rag_files)} files)')
print('Tutor mode:', cfg.get('tutor', {}).get('mode'), '| quantization:', cfg.get('tutor', {}).get('quantization'))
print('TurboQuant base URL:', cfg.get('tutor', {}).get('turboquant_base_url') or 'not configured; tutor will use local TurboQuant cache generation')

cmd = [sys.executable, 'src/rag/build_index.py', '--config', str(CONFIG_PATH)]
print('Running:', ' '.join(str(x) for x in cmd))
subprocess.run(cmd, check=True, cwd=BASE_DIR)

if RUN_TUTOR_DEMO:
    cmd = [
        sys.executable,
        'run_tutor_demo.py',
        '--config',
        str(CONFIG_PATH),
        '--question',
        'I am stuck. How should I start solving this problem?',
        '--skill',
        'linear equations',
        '--mastery',
        '0.42',
    ]
    print('Running:', ' '.join(str(x) for x in cmd))
    subprocess.run(cmd, check=True, cwd=BASE_DIR)
else:
    print('Skipping tutor demo.')


In [ ]:
# ============================================================
# 7. ONNX export, XAI explanations, final eval, benchmark
# ============================================================

import subprocess
import sys

if RUN_ONNX_EXPORT:
    cmd = [sys.executable, 'src/deployment/export_onnx.py', '--config', str(CONFIG_PATH)]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=BASE_DIR)
else:
    print('Skipping ONNX export.')

if RUN_XAI:
    cmd = [sys.executable, 'src/xai/generate_explanations.py', '--config', str(CONFIG_PATH), '--max-examples', '5']
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=BASE_DIR)
else:
    print('Skipping XAI explanations.')

if RUN_FINAL_EVAL:
    checkpoint_path = BASE_DIR / 'checkpoints' / 'best_model.pt'
    prediction_cache = BASE_DIR / 'outputs' / 'evaluation' / 'akt_test_predictions.npz'
    cache_is_fresh = prediction_cache.exists() and (not checkpoint_path.exists() or prediction_cache.stat().st_mtime >= checkpoint_path.stat().st_mtime)
    refresh_graph_data = bool(globals().get('REFRESH_AKT_GRAPH_DATA', False)) or not cache_is_fresh

    if refresh_graph_data:
        if not prediction_cache.exists():
            print('Prediction-level graph data is missing; running one full test evaluation pass.')
        elif not cache_is_fresh:
            print('Prediction-level graph data is older than the best checkpoint; refreshing it.')
        else:
            print('Refreshing prediction-level graph data by request.')
        cmd = [sys.executable, 'src/eval/enhanced_kt_evaluation.py', '--config', str(CONFIG_PATH), '--full-eval']
        if checkpoint_path.exists():
            cmd.extend(['--checkpoint', str(checkpoint_path)])
        print('Running:', ' '.join(str(x) for x in cmd))
        subprocess.run(cmd, check=True, cwd=BASE_DIR)
    else:
        cmd = [sys.executable, 'src/eval/final_eval.py', '--config', str(CONFIG_PATH)]
        print('Running:', ' '.join(str(x) for x in cmd))
        subprocess.run(cmd, check=True, cwd=BASE_DIR)

        cmd = [sys.executable, 'src/eval/enhanced_kt_evaluation.py', '--config', str(CONFIG_PATH)]
        print('Running:', ' '.join(str(x) for x in cmd))
        subprocess.run(cmd, check=True, cwd=BASE_DIR)
else:
    print('Skipping final eval and enhanced evaluation report.')

if RUN_BENCHMARK:
    cmd = [sys.executable, 'src/eval/benchmark_efficiency.py', '--config', str(CONFIG_PATH)]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=BASE_DIR)
else:
    print('Skipping efficiency benchmark.')


In [ ]:
# ============================================================
# 8. Evaluation graphs for report
# ============================================================

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

OUTPUTS_DIR = BASE_DIR / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

saved_figures = []

def save_and_show(fig, filename, title=None):
    path = FIGURES_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches='tight')
    saved_figures.append(str(path))
    if title:
        print(title)
    print('Saved:', path)
    plt.show()
    plt.close(fig)

def read_json(path):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc:
        print('Could not read JSON:', path, exc)
        return None

def numeric_value(value):
    try:
        if value is None or value == '':
            return None
        value = float(value)
        if value != value:
            return None
        return value
    except Exception:
        return None

metrics_path = OUTPUTS_DIR / 'metrics.csv'
training_log_path = OUTPUTS_DIR / 'training_log.json'
metrics_df = None
if metrics_path.exists():
    metrics_df = pd.read_csv(metrics_path)
elif training_log_path.exists():
    training_log = read_json(training_log_path) or {}
    history = training_log.get('history', [])
    if history:
        metrics_df = pd.DataFrame(history)

if metrics_df is not None and not metrics_df.empty:
    print('Loaded training metrics rows:', len(metrics_df))
    if 'epoch' not in metrics_df.columns:
        metrics_df['epoch'] = range(len(metrics_df))

    loss_cols = [c for c in ['train_loss', 'val_loss', 'test_loss'] if c in metrics_df.columns]
    if loss_cols:
        fig, ax = plt.subplots(figsize=(8.5, 5))
        for col in loss_cols:
            ax.plot(metrics_df['epoch'], metrics_df[col], marker='o', linewidth=2, label=col.replace('_', ' ').title())
        ax.set_title('AKT Training and Validation Loss')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('BCE Loss')
        ax.grid(True, alpha=0.25)
        ax.legend()
        save_and_show(fig, 'akt_loss_curve.png', 'Loss curve')

    metric_cols = [c for c in ['val_auc', 'val_accuracy', 'test_auc', 'test_accuracy'] if c in metrics_df.columns]
    if metric_cols:
        fig, ax = plt.subplots(figsize=(8.5, 5))
        for col in metric_cols:
            ax.plot(metrics_df['epoch'], metrics_df[col], marker='o', linewidth=2, label=col.replace('_', ' ').upper())
        ax.set_title('AKT Validation/Test Quality Metrics')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Score')
        ax.set_ylim(0, 1.02)
        ax.grid(True, alpha=0.25)
        ax.legend()
        save_and_show(fig, 'akt_quality_curve.png', 'Quality curve')
else:
    print('No metrics.csv/training_log.json found yet. Run AKT training first for learning curves.')

final_eval_csv = OUTPUTS_DIR / 'final_eval.csv'
final_eval_json = OUTPUTS_DIR / 'final_eval.json'
summary_json = OUTPUTS_DIR / 'akt_training_summary.json'
final_row = {}
if final_eval_csv.exists():
    eval_df = pd.read_csv(final_eval_csv)
    if not eval_df.empty:
        final_row.update(eval_df.iloc[-1].to_dict())
        print('Loaded final eval CSV:', final_eval_csv)
if final_eval_json.exists():
    payload = read_json(final_eval_json) or {}
    if isinstance(payload, dict):
        final_row.update(payload)
if summary_json.exists():
    summary = read_json(summary_json) or {}
    test_metrics = summary.get('test_metrics', {}) if isinstance(summary, dict) else {}
    for key, value in test_metrics.items():
        final_row.setdefault(f'test_{key}', value)
    for key in ['n_questions', 'n_skills', 'epochs_ran', 'dataset_path']:
        if key in summary:
            final_row.setdefault(key, summary[key])

quality_keys = ['test_auc', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_pr_auc', 'val_auc', 'val_accuracy']
quality = {k: numeric_value(final_row.get(k)) for k in quality_keys}
quality = {k: v for k, v in quality.items() if v is not None}
if quality:
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    labels = [k.replace('_', ' ').upper() for k in quality.keys()]
    values = list(quality.values())
    bars = ax.bar(labels, values, color=['#2563eb', '#16a34a', '#7c3aed', '#f59e0b'][:len(values)])
    ax.set_title('Final Student-Model Evaluation')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.02)
    ax.grid(axis='y', alpha=0.25)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 0.015, f'{value:.3f}', ha='center', va='bottom')
    save_and_show(fig, 'final_student_model_scores.png', 'Final student-model scores')
else:
    print('No final AUC/accuracy metrics found yet.')

scale = {k: numeric_value(final_row.get(k)) for k in ['n_questions', 'n_skills', 'epochs_ran']}
scale = {k: v for k, v in scale.items() if v is not None}
if scale:
    fig, ax = plt.subplots(figsize=(7.5, 4.8))
    labels = [k.replace('_', ' ').title() for k in scale.keys()]
    values = list(scale.values())
    bars = ax.bar(labels, values, color=['#334155', '#475569', '#64748b'][:len(values)])
    ax.set_title('Dataset and Model Scale')
    ax.set_ylabel('Count')
    ax.grid(axis='y', alpha=0.25)
    for bar, value in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, value, f'{int(value):,}', ha='center', va='bottom')
    save_and_show(fig, 'dataset_model_scale.png', 'Dataset/model scale')


EVALUATION_DIR = OUTPUTS_DIR / 'evaluation'
enhanced_metrics_csv = EVALUATION_DIR / 'final_test_metrics.csv'
model_comparison_csv = EVALUATION_DIR / 'model_comparison_metrics.csv'
source_standards_csv = EVALUATION_DIR / 'source_evaluation_standards.csv'

try:
    from IPython.display import Image, display
except Exception:
    Image = None
    display = print

if enhanced_metrics_csv.exists():
    enhanced_df = pd.read_csv(enhanced_metrics_csv)
    print('Enhanced final test metrics:', enhanced_metrics_csv)
    display_cols = [
        col for col in ['model', 'auc', 'accuracy', 'precision', 'recall', 'f1', 'pr_auc', 'loss', 'n_tokens', 'tn', 'fp', 'fn', 'tp']
        if col in enhanced_df.columns
    ]
    display(enhanced_df[display_cols] if display_cols else enhanced_df)
else:
    print('Enhanced final test metrics not found yet. Run section 7 first.')

if model_comparison_csv.exists():
    comparison_df = pd.read_csv(model_comparison_csv)
    print('Model comparison metrics:', model_comparison_csv)
    display(comparison_df)

if source_standards_csv.exists():
    source_df = pd.read_csv(source_standards_csv)
    print('Source evaluation standards:', source_standards_csv)
    display(source_df[['source', 'reported_metrics', 'comparability']] if {'source', 'reported_metrics', 'comparability'}.issubset(source_df.columns) else source_df)

evaluation_graphs = [
    EVALUATION_DIR / 'akt_confusion_matrix.png',
    EVALUATION_DIR / 'akt_roc_curve.png',
    EVALUATION_DIR / 'akt_pr_curve.png',
    EVALUATION_DIR / 'model_metric_comparison.png',
]
for graph_path in evaluation_graphs:
    if graph_path.exists():
        print('Evaluation graph:', graph_path)
        saved_figures.append(str(graph_path))
        if Image is not None:
            display(Image(filename=str(graph_path)))
    else:
        print('Missing evaluation graph:', graph_path)

manifest = {'figures_dir': str(FIGURES_DIR), 'saved_figures': saved_figures}
(FIGURES_DIR / 'figures_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Figure manifest:', FIGURES_DIR / 'figures_manifest.json')
if not saved_figures:
    print('No figures created yet. Finish training/evaluation, then rerun this cell.')


In [ ]:
# ============================================================
# 9. Artifact manifest
# ============================================================

import json
from pathlib import Path
import torch

manifest = {
    'base_dir': str(BASE_DIR),
    'config_path': str(CONFIG_PATH),
    'dataset_path': str(DATASET_PATH),
    'organized_datasets_dir': str(ORGANIZED_DATASETS_DIR),
    'rag_docs_dir': str(RAG_DOCS_PATH),
    'cuda_available': bool(torch.cuda.is_available()),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'checkpoint_dir': str(BASE_DIR / 'checkpoints'),
    'best_model': str(BASE_DIR / 'checkpoints' / 'best_model.pt'),
    'last_checkpoint': str(BASE_DIR / 'checkpoints' / 'last_checkpoint.pt'),
    'outputs_dir': str(BASE_DIR / 'outputs'),
    'metrics_csv': str(BASE_DIR / 'outputs' / 'metrics.csv'),
    'training_log': str(BASE_DIR / 'outputs' / 'training_log.json'),
    'onnx_model': str(BASE_DIR / 'outputs' / 'onnx' / 'akt_model.onnx'),
    'final_eval_csv': str(BASE_DIR / 'outputs' / 'final_eval.csv'),
    'evaluation_dir': str(BASE_DIR / 'outputs' / 'evaluation'),
    'enhanced_metrics_csv': str(BASE_DIR / 'outputs' / 'evaluation' / 'final_test_metrics.csv'),
    'prediction_cache': str(BASE_DIR / 'outputs' / 'evaluation' / 'akt_test_predictions.npz'),
    'model_comparison_csv': str(BASE_DIR / 'outputs' / 'evaluation' / 'model_comparison_metrics.csv'),
    'figures_dir': str(BASE_DIR / 'outputs' / 'figures'),
}
(BASE_DIR / 'outputs').mkdir(parents=True, exist_ok=True)
(BASE_DIR / 'outputs' / 'local_artifact_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))
